### Vertex AI 에서 vLLM for GPU 를 이용한 sLLM 모델 서빙 환경 구성 실습

In [ ]:
import google.auth
credentials, PROJECT_ID = google.auth.default()
PROJECT_ID

Vertex AI 팀에서 릴리즈 하는 vLLM 컨테이너 정보 및 배포 리전 설정

Vertex AI vLLM 의 추가 기능: 

https://docs.cloud.google.com/vertex-ai/generative-ai/docs/open-models/vllm/use-vllm?hl=ko#google-vertex-ai-vllm-customizations-enhancing-performance-and-integration

In [ ]:
#Basic configuration
import os
import logging
import json
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
logging.basicConfig(level=logging.INFO)
SERVICE_ACCOUNT = f"{PROJECT_ID}@{PROJECT_ID}.iam.gserviceaccount.com"   #SA to access gcs

배포할 모델 및 인프라 정의

In [ ]:
#Model configuration
DEPLOY_TIMEOUT = 3600
LOCATION = "asia-northeast3"
vLLM_DOCKER_URI = "us-docker.pkg.dev/agent-platform-mg-public/containers/pytorch-vllm-serve:model-garden.pytorch-vllm-serve-release_20261006.00_p0"
MODEL_NAME = "Gemma Embedding 2"
#Can have your own path
MODEL_ID = "google/embeddinggemma-2"
MODEL_PATH = "gs://vertex-model-garden-restricted-us/embeddinggemma/embeddinggemma-2"
MAX_MODEL_LEN = 8192
MACHINE_TYPE = f"a3-edgegpu-8g" #a2-highgpu-4g, g2-standard-48, a3-edgegpu-8g
ACCELERATOR_TYPE = "NVIDIA_H100_80GB" #NVIDIA_TESLA_A100, NVIDIA_L4, NVIDIA_H100_80GB
ACCELERATOR_COUNT = 1

vLLM 과 LLM 모델을 묶어 Vertex AI Model Registry 에 등록

In [ ]:
#Upload model pacakge to vertex model repository
from google.cloud import aiplatform
model_v1 = aiplatform.Model.upload(
    location=LOCATION,
    display_name = MODEL_NAME,
    artifact_uri = MODEL_PATH,
    #For direct container upload rather then use LocalModel
    serving_container_image_uri=vLLM_DOCKER_URI,    
    serving_container_predict_route="/v1/embeddings",
    serving_container_health_route="/ping",
    serving_container_ports=[8080],
    serving_container_environment_variables={
        #"VLLM_ALLOW_LONG_MAX_MODEL_LEN": "1",
        #"VLLM_USE_V1": "1",
        #"HF_TOKEN": hf_token
        "MODEL_ID": MODEL_ID,
        "DEPLOY_SOURCE": "API_NATIVE_MODEL"
    },
    serving_container_args=["python",
                            "-m",
                            "vllm.entrypoints.api_server",
                            "--port=8080",
                            "--host=0.0.0.0",
                            "--runner=pooling",
                            f"--model={MODEL_ID}",
                            f"--max-model-len={MAX_MODEL_LEN}",
                            ],
)

외부 서비스를 위한 Endpoint 생성

In [ ]:
from google.cloud import aiplatform
endpoint = aiplatform.Endpoint.create(
    display_name=f"{MODEL_NAME} public dedicated test endpoint",
    #labels={"sample-key": "sample-value"},
    location=LOCATION,
    dedicated_endpoint_enabled=True,
)

Endpoint 내 Vertex AI Model Registry 에 등록된 모델 배포 (최초 배포시 최소 30분 이상 시간 필요)

In [ ]:
#Public endpoint with dedicated network
response = endpoint.deploy(
    model = model_v1,
    machine_type=MACHINE_TYPE,
    min_replica_count=8,
    max_replica_count=8,
    #service_account=SERVICE_ACCOUNT,
    #traffic_percentage=50
    #traffic_split={'a':50, 'b':50}
    #Configs for GPU
    #accelerator_type="NVIDIA_L4",
    #machine_type="a2-highgpu-1g",
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=1,
    #required_replica_count=ACCELERATOR_COUNT,
    #deploy_request_timeout=DEPLOY_TIMEOUT
)

Endpoint class creation. Check Endpoint address through Google Cloud console

In [ ]:
import json
import time
from google.cloud import aiplatform

#Create endpoint class
endpoint = aiplatform.Endpoint("projects/sandbox-373102/locations/asia-northeast3/endpoints/3978607014363267072")

body = {
    "input": [
        #"task: search result | query: What causes the northern lights?",
        "title: none | text: The northern lights are caused by charged particles from the sun.",
    ]
}

Stress test by running 100 times

In [ ]:
start_time = time.perf_counter()
for i in range(0, 100):
    response = endpoint.raw_predict(
        body=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        use_dedicated_endpoint=True,  # False for a shared endpoint
    )
    embeddings = [item["embedding"] for item in response.json()["data"]]
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")
len(embeddings[0])

Interleaved Multimodal input test

In [ ]:
#Example for interleaving
body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Waterproof running shoes. "},
                {"type": "image_url", "image_url": {"url": "https://example.com/shoe.jpg"}}
            ],
        }
    ]
}